# Week 4 Assignment: Pretrained GloVe Word Embeddings

## Word-vector arithmetic and semantic relationships

This notebook loads pretrained GloVe vectors, computes the required analogy

```text
vec(woman) - vec(man) + vec(king)
```

and retrieves the nearest words using cosine similarity. Several additional analogies are tested and interpreted.

## 1. Selected pretrained embedding file

The selected resource is the 50-dimensional GloVe 6B file from Zenodo record 4925376. This is the Wikipedia 2014 + Gigaword 5 collection. The 50d archive is used because it is substantially smaller than the 100d, 200d, and 300d versions while still being adequate for demonstrating vector arithmetic.

The archive is downloaded only if it is not already present. Its 50-dimensional vectors are read directly with `numpy` from the included binary word2vec-format file, so no extra embedding library is needed.

In [6]:
from pathlib import Path
import urllib.request
import zipfile
import numpy as np

# Download the 50-dimensional pretrained GloVe vectors if needed.
glove_url = 'https://zenodo.org/records/4925376/files/glove.6B.50d.zip?download=1'
data_dir = Path('glove_data')
data_dir.mkdir(exist_ok=True)
zip_path = data_dir / 'glove.6B.50d.zip'
embedding_path = data_dir / 'glove.6B.50d'

if not embedding_path.exists():
    if not zip_path.exists():
        print('Downloading GloVe 50d archive...')
        urllib.request.urlretrieve(glove_url, zip_path)
    with zipfile.ZipFile(zip_path) as archive:
        member = 'glove.6B.50d'
        if member not in archive.namelist():
            raise FileNotFoundError(f'{member} was not found in the downloaded archive.')
        archive.extract(member, data_dir)
else:
    print('Using existing GloVe vectors.')

print('Embedding file:', embedding_path)


Using existing GloVe vectors.
Embedding file: glove_data\glove.6B.50d


## 2. Load the vectors

Each word is represented by a 50-dimensional vector. The downloaded file is a pretrained GloVe resource stored in binary word2vec format. The vector space is learned from word co-occurrence statistics: words used in similar contexts tend to have similar vector directions. This is why cosine similarity is useful for comparing word vectors.

In [7]:
# Read the downloaded GloVe vectors in binary word2vec format.
# The archive stores one word followed by 50 float32 values for each line.
words = []
vectors = []

with embedding_path.open('rb') as file:
    vocabulary_size, vector_size = map(int, file.readline().split())
    for _ in range(vocabulary_size):
        word_bytes = bytearray()
        while True:
            character = file.read(1)
            if character == b' ':
                break
            if not character:
                raise EOFError('Unexpected end of the embedding file.')
            word_bytes.extend(character)
        words.append(word_bytes.decode('utf-8'))
        vectors.append(np.fromfile(file, dtype=np.float32, count=vector_size))

word_vectors = np.vstack(vectors)
word_index = {word: index for index, word in enumerate(words)}
normalized_vectors = word_vectors / np.linalg.norm(word_vectors, axis=1, keepdims=True)

print('Vocabulary size:', len(words))
print('Vector dimension:', word_vectors.shape[1])
print('First vocabulary items:', words[:10])


Vocabulary size: 400000
Vector dimension: 50
First vocabulary items: ['the', ',', '.', 'of', 'to', 'and', 'in', 'a', '"', "'s"]


## 3. Cosine similarity

Cosine similarity compares the angle between two vectors rather than their magnitude:

$$\operatorname{cosine}(a,b)=\frac{a\cdot b}{||a||\,||b||}$$

A value close to 1 means that the vectors point in similar directions. For analogy search, the target vector is compared against all vocabulary vectors, and the words with the highest cosine scores are returned.

In [8]:
def vector(word):
    if word not in word_index:
        raise KeyError(f'{word!r} is not in the GloVe vocabulary')
    return word_vectors[word_index[word]]


def nearest_words(target_vector, top_n=10, exclude=()):
    target = target_vector / np.linalg.norm(target_vector)
    scores = normalized_vectors @ target
    scores = scores.copy()
    for word in exclude:
        if word in word_index:
            scores[word_index[word]] = -np.inf
    best_indices = np.argsort(scores)[::-1][:top_n]
    return [(words[index], float(scores[index])) for index in best_indices]


## 4. Required analogy: woman − man + king

The requested target vector is calculated explicitly. The words used to construct the target are excluded from the nearest-neighbour list so that the answer reflects a new nearby word rather than simply returning one of the input words.

In [9]:
man = vector('man')
woman = vector('woman')
king = vector('king')

target_vector = woman - man + king
required_result = nearest_words(
    target_vector, top_n=10, exclude=('man', 'woman', 'king')
)

print('Nearest words to vec(woman) - vec(man) + vec(king):')
for word, score in required_result:
    print(f'{word:20s} cosine similarity = {score:.4f}')

Nearest words to vec(woman) - vec(man) + vec(king):
queen                cosine similarity = 0.8610
daughter             cosine similarity = 0.7685
prince               cosine similarity = 0.7641
throne               cosine similarity = 0.7635
princess             cosine similarity = 0.7513
elizabeth            cosine similarity = 0.7506
father               cosine similarity = 0.7314
kingdom              cosine similarity = 0.7296
mother               cosine similarity = 0.7280
son                  cosine similarity = 0.7280


### Interpretation of the required result

The operation attempts to remove the semantic direction associated with `man` and add the corresponding direction associated with `woman` to the vector for `king`. In a well-trained embedding space, the nearest result is expected to be a feminine counterpart of `king`, typically `queen` or a closely related royal term.

This does not mean that the model understands a formal rule from a dictionary. It has learned geometric regularities from word contexts. The result can vary with the embedding version, dimensionality, vocabulary, and normalization.

## 5. Additional semantic analogies

The following examples test several types of relationships: geography, family/gender, professions, and comparative or grammatical patterns. The code prints the nearest neighbours so the actual results from the selected GloVe file are visible in the notebook.

In [10]:
analogy_examples = [
    ('italy', 'france', 'paris', 'geography: Paris is to France as ? is to Italy'),
    ('woman', 'man', 'brother', 'family/gender: brother is to man as ? is to woman'),
    ('walk', 'walking', 'swimming', 'grammar: walking is to walk as swimming is to ?'),
    ('big', 'bigger', 'smaller', 'comparison: bigger is to big as smaller is to ?'),
]

for positive_word, negative_word, anchor_word, description in analogy_examples:
    target = vector(anchor_word) + vector(positive_word) - vector(negative_word)
    result = nearest_words(
        target, top_n=5,
        exclude=(positive_word, negative_word, anchor_word)
    )
    print('\n' + description)
    print(f'  vec({anchor_word}) + vec({positive_word}) - vec({negative_word})')
    for word, score in result:
        print(f'  {word:20s} {score:.4f}')


geography: Paris is to France as ? is to Italy
  vec(paris) + vec(italy) - vec(france)
  rome                 0.8379
  milan                0.7715
  turin                0.7603
  venice               0.7575
  madrid               0.7458

family/gender: brother is to man as ? is to woman
  vec(brother) + vec(woman) - vec(man)
  daughter             0.9374
  wife                 0.9135
  mother               0.8992
  father               0.8937
  cousin               0.8920

grammar: walking is to walk as swimming is to ?
  vec(swimming) + vec(walk) - vec(walking)
  pool                 0.8194
  swim                 0.8134
  indoor               0.7954
  rowing               0.7764
  olympic              0.7463

comparison: bigger is to big as smaller is to ?
  vec(smaller) + vec(big) - vec(bigger)
  small                0.8801
  large                0.8475
  other                0.8320
  well                 0.8090
  few                  0.8049


The result `rome` is a strong result because Paris is the capital of France, while Rome is the capital of Italy. The vector arithmetic approximately transfers the capital–country relationship from France to Italy.


The expected result would be `sister`, but the nearest words are other female or family-related terms. This indicates that the learned gender direction is approximate and that the embedding captures broad contextual similarity rather than an exact family-role rule.


The word `swim` appears near the top, so the model partially captures the grammatical relation. However, `pool` is ranked first because swimming frequently occurs in a pool-related context. This shows the interaction between grammatical and topical information.


## 6. Reflection on the additional examples

The geography example tests whether the embedding captures a country–capital association. The family example tests a gender transformation while preserving a family relationship. The grammatical examples test whether the vector space contains directions associated with inflectional changes such as `walk`/`walking` or `big`/`bigger`.

The nearest result is not guaranteed to be the exact expected word. GloVe is trained from distributional evidence, not from a symbolic knowledge base. It can therefore return a related country, city, family term, or grammatical form. Results are also influenced by corpus frequency and may reflect social biases present in the training text.

A strong result is one where the returned word preserves both parts of the relationship: the semantic relation being added and the relation being removed. A weak or surprising result is still informative because it shows the limitations of representing language with a fixed geometric vector space.

## Conclusion

Pretrained GloVe embeddings allow us to reuse word representations learned from a large text corpus. Vector arithmetic can reveal approximate semantic and syntactic relationships because related transformations are sometimes encoded as directions in the embedding space. The required operation and the additional examples demonstrate both the usefulness and the limitations of this approach.

### Source
Bernhard Liebl, *GloVe 6B Vectors*, Zenodo, record 4925376: https://zenodo.org/records/4925376